# Notebook 10 — Spatial Risk Map

Generate a choropleth map showing predicted stranding counts per region for a target week.

**Reads:** `models/lgbm_model.pkl`, `plankton_imputed_lookup.parquet`, `final_dataset.parquet`  
**Writes:** `figures/risk_map_[YYYY-WW].png`

In [ ]:
import pandas as pd
import numpy as np
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from datetime import datetime, timedelta

from se_coast_strandings.contextual_data.lunar_phases import moon_age, moon_phase
from se_coast_strandings.transformations import make_cyclic, make_cyclic_season
from se_coast_strandings.contextual_data.plankton_abundance import DEFAULT_REGIONS

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
FIGURES_DIR = Path("../figures")
REFERENCE_DIR = Path("../data/reference")
FIGURES_DIR.mkdir(exist_ok=True)

## Load Model and Data

In [ ]:
with open(MODELS_DIR / "lgbm_model.pkl", "rb") as f:
    model = pickle.load(f)

weekly = pd.read_parquet(PROCESSED_DIR / "final_dataset.parquet")
plankton = pd.read_parquet(PROCESSED_DIR / "plankton_imputed_lookup.parquet")
plankton = plankton.rename(columns={"ds": "week_start", "yhat": "plankton_density"})
plankton["week_start"] = pd.to_datetime(plankton["week_start"])

print(f"Model loaded. Historical data: {len(weekly)} rows")

## Set Target Week

In [ ]:
# Default: next Monday from today
today = pd.Timestamp.now()
days_until_monday = (7 - today.weekday()) % 7
if days_until_monday == 0:
    days_until_monday = 7
target_week = (today + timedelta(days=days_until_monday)).normalize()

# Override manually if needed:
# target_week = pd.Timestamp("2024-03-04")

iso_year, iso_week, _ = target_week.isocalendar()
print(f"Target week: {target_week.strftime('%Y-%m-%d')} (ISO {iso_year}-W{iso_week:02d})")

## Build Feature Rows for Each Region

In [ ]:
regions = [label for label, _, _ in DEFAULT_REGIONS]

# Get the feature columns the model expects
FEATURE_COLS = model.feature_name() if hasattr(model, 'feature_name') else [
    "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    "season_sin", "season_cos", "moon_age",
    "temperature_2m_max_0_days_prior_mean",
    "temperature_2m_min_0_days_prior_mean",
    "temperature_2m_max_0_days_prior_max",
    "temp_delta_day0_mean",
    "plankton_density",
    "stranding_count_lag_1", "stranding_count_lag_52",
    "region_SC", "region_NC-south", "region_NC-north", "region_VA",
]

rows = []
for region in regions:
    row = {}

    # Cyclic time features
    month_s = pd.Series([target_week.month])
    doy_s = pd.Series([target_week.dayofyear])
    date_s = pd.Series([target_week])

    m_sin, m_cos = make_cyclic(month_s, 12, name="month")
    row["month_sin"] = float(m_sin.iloc[0])
    row["month_cos"] = float(m_cos.iloc[0])

    d_sin, d_cos = make_cyclic(doy_s, 365, name="dayofyear")
    row["dayofyear_sin"] = float(d_sin.iloc[0])
    row["dayofyear_cos"] = float(d_cos.iloc[0])

    s_sin, s_cos = make_cyclic_season(date_s, name="season")
    row["season_sin"] = float(s_sin.iloc[0])
    row["season_cos"] = float(s_cos.iloc[0])

    # Moon features
    row["moon_age"] = moon_age(target_week)

    # Plankton density (nearest week from lookup)
    region_plankton = plankton[plankton["region"] == region]
    if not region_plankton.empty:
        diffs = (region_plankton["week_start"] - target_week).abs()
        nearest_idx = diffs.idxmin()
        row["plankton_density"] = float(region_plankton.loc[nearest_idx, "plankton_density"])
    else:
        row["plankton_density"] = 0.0

    # Weather: use last observed week as proxy (MVP)
    region_hist = weekly[weekly["region"] == region].sort_values("week_start")
    if not region_hist.empty:
        last_row = region_hist.iloc[-1]
        for col in ["temperature_2m_max_0_days_prior_mean",
                     "temperature_2m_min_0_days_prior_mean",
                     "temperature_2m_max_0_days_prior_max",
                     "temp_delta_day0_mean"]:
            if col in last_row.index:
                row[col] = float(last_row[col]) if pd.notna(last_row[col]) else 0.0

    # Lag features: use last known values
    if not region_hist.empty:
        row["stranding_count_lag_1"] = float(region_hist.iloc[-1]["stranding_count"])
        if len(region_hist) >= 52:
            row["stranding_count_lag_52"] = float(region_hist.iloc[-52]["stranding_count"])
        else:
            row["stranding_count_lag_52"] = float(region_hist["stranding_count"].mean())

    # Region one-hot
    for r in regions:
        row[f"region_{r}"] = 1.0 if r == region else 0.0

    row["region"] = region
    rows.append(row)

feature_df = pd.DataFrame(rows)

# Ensure all expected columns exist
for col in FEATURE_COLS:
    if col not in feature_df.columns:
        feature_df[col] = 0.0

print(feature_df[["region"] + [c for c in FEATURE_COLS if c in feature_df.columns]])

## Predict

In [ ]:
predictions = model.predict(feature_df[FEATURE_COLS])
feature_df["predicted_count"] = predictions

print(f"\nPredicted strandings for week of {target_week.strftime('%Y-%m-%d')}:")
for _, row in feature_df.iterrows():
    print(f"  {row['region']:>10s}: {row['predicted_count']:.1f}")

## Generate Choropleth Risk Map

In [ ]:
# Load state boundaries
states = gpd.read_file(REFERENCE_DIR / "cb_2018_us_state_5m.shp")
se_states = states[states["NAME"].isin(["Virginia", "North Carolina", "South Carolina"])]

# Map regions to latitude bands for visualization
region_polys = []
for label, lat_min, lat_max in DEFAULT_REGIONS:
    pred = feature_df[feature_df["region"] == label]["predicted_count"].values[0]
    region_polys.append({
        "region": label,
        "lat_min": lat_min,
        "lat_max": lat_max,
        "lat_center": (lat_min + lat_max) / 2,
        "predicted_count": pred,
    })

region_df = pd.DataFrame(region_polys)

# Create the risk map
fig, ax = plt.subplots(figsize=(12, 10))

# Plot state boundaries
se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

# Color regions by predicted count
cmap = plt.cm.YlOrRd
norm = plt.Normalize(
    vmin=region_df["predicted_count"].min(),
    vmax=max(region_df["predicted_count"].max(), 1),
)

for _, row in region_df.iterrows():
    color = cmap(norm(row["predicted_count"]))
    ax.axhspan(row["lat_min"], row["lat_max"], xmin=0, xmax=1,
               alpha=0.4, color=color, zorder=2)
    ax.text(-77.5, row["lat_center"], f"{row['region']}\n{row['predicted_count']:.1f}",
            ha="center", va="center", fontsize=12, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
            zorder=3)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, label="Predicted Weekly Strandings", shrink=0.6)

ax.set_xlim(-82, -74)
ax.set_ylim(31.5, 38.5)
ax.set_title(f"Predicted Marine Mammal Strandings\nWeek of {target_week.strftime('%Y-%m-%d')} (ISO {iso_year}-W{iso_week:02d})",
             fontsize=14, pad=15)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

out_path = FIGURES_DIR / f"risk_map_{iso_year}-W{iso_week:02d}.png"
plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"\nSaved: {out_path}")